In [1]:
import pandas as pd
import csv
import pandas_gbq
import time
from datetime import datetime
import os
from google.oauth2 import service_account
from google.cloud import bigquery
import re

In [2]:
CREDS = '../converge-database-0331482f2ee5.json'
client = bigquery.Client.from_service_account_json(json_credentials_path=CREDS)

In [63]:
seriatim = pd.DataFrame()
policy = pd.DataFrame()
notional = pd.DataFrame()
withdrawals = pd.DataFrame()
premiums = pd.DataFrame()
commissions = pd.DataFrame()
#set_month = '202301'

In [64]:
file_path = "I:New Structure/Actuarial New/Database/SILAC Settlements/2026/202603 Converge Teton Settlement file.xlsx"

In [65]:
seriatim = pd.read_excel(file_path, sheet_name='Seriatim')
withdrawals = pd.read_excel(file_path, sheet_name='Withdrawals')
premiums = pd.read_excel(file_path, sheet_name='Premiums')
notional = pd.read_excel(file_path, sheet_name='Notional')
commissions = pd.read_excel(file_path, sheet_name='Commissions')
deaths = pd.read_excel(file_path, sheet_name='Monthly Deaths')

In [66]:
credit_dictionary ={'Barclays Atlas 5 Point-to-Point PR' : 'ATLAS-AP2PPR', 'S&P 500 RavenPack AI Point-to-Point PR' : 'RVP-AP2PPR', 
                    'S&P 500 Monthly Average PR' : 'MAPR', 'S&P 500 Monthly Point-to-Point Cap': 'MP2PC', 'S&P 500 Point-to-Point Cap': 'AP2PC' ,
                    'S&P 500 Point-to-Point PR' : 'AP2PPR', 'NDX Generations 5 Point-to-Point PR': 'NASDAQ-AP2PPR', 'Barclays Atlas 5 Point-to-Point Spread' : 'ATLAS-AP2PS' ,
                   'Fixed Interest' : 'FI', 'NDX Generations 5 Point-to-Point Spread' : 'NASDAQ-AP2PS', 'S&P 500 RavenPack AI Point-to-Point Spread' : 'RVP-AP2PS' ,
                   'S&P 500 Monthly Average Cap' : 'MAC', 'S&P 500 Monthly Average Spread' : 'MAS',
                   'S&P 500 Duo Swift Point-to-Point PR' : 'SPDS-AP2PPR', 'CS RavenPack AI Point-to-Point PR' : 'RVP-AP2PPR', 'CS RavenPack AI Point-to-Point Spread': 'RVP-AP2PS',
                   'S&P 500 RavenPack AI Point-to-Point Sprd' : 'RVP-AP2PS'}

#Credit strategy changes every month

In [67]:
def clean_columns(df, *, space_replacement="", replace_parens=False):
    cols = (
        df.columns
        .str.strip()
        .str.lower()
        .map(lambda x: x.replace(" ", space_replacement))
    )
    
    if replace_parens:
        cols = (
            cols
            .map(lambda x: x.replace("(", "_"))
            .map(lambda x: x.replace(")", "_"))
        )
    
    df.columns = cols
    return df

In [68]:
seriatim = clean_columns(seriatim, space_replacement="_")
withdrawals = clean_columns(withdrawals, replace_parens=True)
premiums = clean_columns(premiums)
notional = clean_columns(notional)
commissions = clean_columns(commissions)
deaths = clean_columns(deaths)

In [69]:
seriatim['credit_id'] = seriatim['creditstrategy'].map(credit_dictionary)

In [70]:
#Getting columns only for the policy 
policy= seriatim.iloc[:,0:19]
# Add a new column named 'credit_id'
policy['credit_id'] = policy['creditstrategy'].map(credit_dictionary)
policy['creditstrategy'] = seriatim['creditstrategy']
policy.rider_spread = policy.rider_spread.astype("int64")

In [71]:
seriatim = seriatim.astype({"mtd_index_interest": "int",  "currentbonusrecoverypercentage" : "float", 
                            "strategy_ytd_nursing_care_withdrawals" : "float","strategy_ytd_home_health_care_withdrawals" : "float", 
                            "policy_ytd_cumulative_withdrawal": "int", "policy_ytd_nursing_care_withdrawals" : "int",
                            "policy_ytd_home_health_care_withdrawals":"float", "total_cumulative_withdrawal" : "int", 
                            "total_nursing_care_withdrawals" : "float", "total_home_health_care_withdrawals" : "float",
                            "strategy_ytd_terminal_illness_withdrawals" : "int", "strategy_ytd_cumulative_withdrawal" : "int", "policy_ytd_nursing_care_withdrawals": "float",
                           "total_terminal_illness_withdrawals" : "int" , "mtd_index_interest": "int","currentbonusrecoverypercentage" : "float",
                           "policy_ytd_terminal_illness_withdrawals" : "int"})

In [72]:
#Get the seriatim values after policy 
seriatim.drop(['product','plan', 'rateversion', 'stateofsale', 'taxqualstatus', 'issueyear', 'issuemonth', 'issueday',
              'issueage', 'ownerresidentstate', 'ownergender' ,'annuitant_issue_age', 'annuitant_gender',
              'joint_annuitant_issue_age', 'joint_annuitant_gender', 'riders', 'rider_spread'], axis=1, inplace=True, errors="ignore")
#seriatim.drop(['converge'], axis=1)
seriatim.rename(columns = {'joint/single_payment': 'joint_single_payment'}, inplace=True)
#premiums['deleteddate'] = pd.to_datetime(premiums['deleteddate'])

In [73]:
withdrawals = withdrawals.drop(['policyid','product', 'plan', 'approvaldate', 'policyissuestate' ], errors='ignore', axis=1)
premiums = premiums.drop(['policyid', 'product', 'approvaldate', 'policyissuestate','ownerissueage'], errors='ignore', axis=1)
notional = notional.drop(['polid','product_conf', 'rateversion', 'product', 'planlength', 'state','creditingstrategyname'], axis=1)
commissions = commissions.drop(['policyid','product','plan', 'approvaldate', 'policyissuestate'], errors='ignore', axis=1)
deaths = deaths.drop(['product', 'plan', 'withdrawaltype'],errors = 'ignore', axis=1)

In [74]:
withdrawals['totalpolicypremiumreceived'] = withdrawals['totalpolicypremiumreceived'].astype("float")
withdrawals['transactiondate'] = withdrawals['transactiondate'].astype("datetime64[ns]")

In [75]:
withdrawals = withdrawals.iloc[:, 0:36]

In [76]:
withdrawals = withdrawals.dropna(subset=['policynumber'], axis=0)

In [77]:
#get seriatim month from transaction date
get_seriatim_month = withdrawals['transactiondate'].apply(lambda x: x.strftime('%Y%m')) 
get_seriatim_month = get_seriatim_month[0]
get_seriatim_month

'202603'

In [78]:
product =policy["policynumber"].astype("str").str[0][0]+"%"

In [79]:
seriatim.rename(columns = {'converge_-_teton': 'converge'}, inplace=True)
withdrawals = withdrawals.drop(['converge-denali'], axis=1, errors="ignore")
commissions = commissions.drop(['converge-denali'], axis=1, errors="ignore")
premiums = premiums.drop(['converge-denali'], axis=1, errors="ignore")
notional = notional.drop(['converge-denali'], axis=1, errors="ignore")

### Header change codes here

In [80]:
seriatim.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48805 entries, 0 to 48804
Data columns (total 64 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   policynumber                               48805 non-null  object 
 1   creditstrategy                             48805 non-null  object 
 2   optionbudget                               48805 non-null  float64
 3   totalinitpremium                           48805 non-null  float64
 4   totaladdlpremium                           48805 non-null  float64
 5   totalpolicypremiums                        48805 non-null  float64
 6   bonus_%                                    48805 non-null  float64
 7   totalinitprembonus                         48805 non-null  float64
 8   totaladdlprembonus                         48805 non-null  float64
 9   totalpolicypremiumbonus                    48805 non-null  float64
 10  totalpolicyiav        

In [81]:
seriatim.rename(columns = {"payout/lifetime_withdrawal_type" : "joint_single_payment", "lifetimelevsingle50" : "lifetimesingle50", "lifetimelevsingle60" : "lifetimesingle60", 
                           "lifetimelevsingle70" : "lifetimesingle70", "lifetimelevsingle80" : "lifetimesingle80", 
                           "lifetimelevjoint50" : "lifetimejoint50", "lifetimelevjoint60" : "lifetimejoint60", "lifetimelevjoint70" : "lifetimejoint70", "lifetimelevjoint80" : "lifetimejoint80"
                          ,"total_premium_bonus_%": "total_premium_bonus_percent", "lifetime_withdrawals_elected_date" : "lifetimewithdrawalselecteddate", 
                           "wellness_withdrawals_elected_date": "wellnesswithdrawalselecteddate", "wellness_withdrawals_termination_date" :"wellnesswithdrawalstermdate"}, inplace=True)

In [82]:
if product =='D%':
    seriatim = seriatim.astype({"lifetimewithdrawalselecteddate" : "datetime64[ns]", "wellnesswithdrawalselecteddate" : "datetime64[ns]", "wellnesswithdrawalstermdate" : "datetime64[ns]" })

In [83]:
def to_upload (data, table_name,get_seriatim_month):
    df = pd.DataFrame()    
    df = data
    if table_name not in ("premiums", "commissions", "notional"):
        df['set_month'] = get_seriatim_month 
    print('start pushing policy data')
    start = time.time()
    df.to_gbq("converge-database.denali."+table_name,
                 if_exists='append',
                 project_id="converge-database") 
    end = time.time()
    logs(get_seriatim_month, file_path, str(table_name))
    print("time took to upload " + str(end - start))
    print('success')

In [84]:
def logs(get_seriatim_month, file_path, sheetname):
    log = pd.DataFrame()
    #creating logs and the timestamp in it. 
    log['time'] = [pd.Timestamp(datetime.now())]
    log['file'] = file_path
    log['sheetname']=sheetname
    log.to_gbq("converge-database.denali.logs",
                   if_exists='append',
                   project_id="converge-database")

In [85]:
policy

,policynumber,product,plan,riders,rider_spread,rateversion,creditstrategy,stateofsale,taxqualstatus,issueyear,issuemonth,issueday,issueage,ownerresidentstate,ownergender,annuitant_issue_age,annuitant_gender,joint_annuitant_issue_age,joint_annuitant_gender,credit_id
0,T000004959,Teton,10,Not Applicable (N/A),0,8.70,Fixed Interest,PA,IRA,2020,12,30,55,PA,Female,55,Female,NaN,NaN,FI
1,T000004959,Teton,10,Not Applicable (N/A),0,8.70,S&P 500 Point-to-Point Cap,PA,IRA,2020,12,30,55,PA,Female,55,Female,NaN,NaN,AP2PC
2,T000005010,Teton,10,Not Applicable (N/A),0,9.90,S&P 500 Monthly Average PR,GA,IRA,2021,3,10,46,GA,Female,46,Female,NaN,NaN,MAPR
3,T000005010,Teton,10,Not Applicable (N/A),0,9.90,Barclays Atlas 5 Point-to-Point PR,GA,IRA,2021,3,10,46,GA,Female,46,Female,NaN,NaN,ATLAS-AP2PPR
4,T000005100,Teton,7,Not Applicable (N/A),0,8.70,S&P 500 Point-to-Point PR,CT,Non-Qualified,2020,12,23,66,CT,Female,66,Female,NaN,NaN,AP2PPR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48800,TB00055502,Teton Bonus,14,Not Applicable (N/A),0,12.18,S&P 500 Point-to-Point Cap,GA,Non-Qualified,2022,12,21,51,GA,Male,51,Male,NaN,NaN,AP2PC
48801,TB00055504,Teton Bonus,14,Not Applicable (N/A),0,12.18,NDX Generations 5 Point-to-Point PR,GA,Non-Qualified,2022,12,21,48,FL,Female,48,Female,NaN,NaN,NASDAQ-AP2PPR
48802,TB00055504,Teton Bonus,14,Not Applicable (N/A),0,12.18,Barclays Atlas 5 Point-to-Point PR,GA,Non-Qualified,2022,12,21,48,FL,Female,48,Female,NaN,NaN,ATLAS-AP2PPR
48803,TB00067493,Teton Bonus,14,Not Applicable (N/A),0,9.80,Barclays Atlas 5 Point-to-Point PR,TN,IRA,2021,3,3,59,TN,Male,59,Male,NaN,NaN,ATLAS-AP2PPR


In [86]:
to_upload(policy,"policy",get_seriatim_month)

start pushing policy data


100%|██████████| 1/1 [00:00<?, ?it/s]

time took to upload 3.966109037399292
success


In [87]:
seriatim = seriatim.iloc[:, 0:87]

In [88]:
seriatim

,policynumber,creditstrategy,optionbudget,totalinitpremium,totaladdlpremium,totalpolicypremiums,bonus_%,totalinitprembonus,totaladdlprembonus,totalpolicypremiumbonus,...,initialwithdrawalcharge,currentwithdrawalcharge,currentbonusrecoverypercentage,mgv_premium,mgv_fixed,mgv_index,reinsurancecode,reserves2,converge,credit_id
0,T000004959,Fixed Interest,0.0225,66434.52,0.45,66434.97,0.0,0.0,0.0,0.0,...,0.0930,0.0475,0.0,0.875,0.0100,0.01,AG,34445.750000,0.10,FI
1,T000004959,S&P 500 Point-to-Point Cap,0.0225,66434.52,0.45,66434.97,0.0,0.0,0.0,0.0,...,0.0930,0.0475,0.0,0.875,0.0100,0.01,AG,34259.695312,0.10,AP2PC
2,T000005010,S&P 500 Monthly Average PR,0.0265,49442.07,0.00,49442.07,0.0,0.0,0.0,0.0,...,0.1200,0.0800,0.0,0.875,0.0100,0.01,AJ,10795.020508,0.10,MAPR
3,T000005010,Barclays Atlas 5 Point-to-Point PR,0.0265,49442.07,0.00,49442.07,0.0,0.0,0.0,0.0,...,0.1200,0.0800,0.0,0.875,0.0100,0.01,AJ,43180.082031,0.10,ATLAS-AP2PPR
4,T000005100,S&P 500 Point-to-Point PR,0.0231,400000.00,200000.00,600000.00,0.0,0.0,0.0,0.0,...,0.1250,0.0800,0.0,0.875,0.0100,0.01,AC,654223.625000,0.10,AP2PPR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48800,TB00055502,S&P 500 Point-to-Point Cap,0.0335,105000.00,0.00,105000.00,0.1,10500.0,0.0,10500.0,...,0.1475,0.1175,1.0,0.875,0.0175,0.01,AHZ,55533.386719,0.15,AP2PC
48801,TB00055504,NDX Generations 5 Point-to-Point PR,0.0335,105000.00,0.00,105000.00,0.1,10500.0,0.0,10500.0,...,0.1475,0.1175,1.0,0.875,0.0175,0.01,AHZ,54168.847656,0.15,NASDAQ-AP2PPR
48802,TB00055504,Barclays Atlas 5 Point-to-Point PR,0.0335,105000.00,0.00,105000.00,0.1,10500.0,0.0,10500.0,...,0.1475,0.1175,1.0,0.875,0.0175,0.01,AHZ,54168.847656,0.15,ATLAS-AP2PPR
48803,TB00067493,Barclays Atlas 5 Point-to-Point PR,0.0228,32795.01,0.00,32795.01,0.1,3279.5,0.0,3279.5,...,0.1475,0.1000,1.0,0.875,0.0100,0.01,AAI,34968.125000,0.10,ATLAS-AP2PPR


In [89]:
to_upload(seriatim,"seriatim_values",get_seriatim_month)

start pushing policy data


100%|██████████| 1/1 [00:00<?, ?it/s]

time took to upload 5.902164936065674
success


In [90]:
notional.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13497 entries, 0 to 13496
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   policynumber            13497 non-null  object        
 1   optionbudget            13497 non-null  float64       
 2   issuedate               13497 non-null  datetime64[ns]
 3   rider                   13497 non-null  object        
 4   reinsurancecode         13497 non-null  object        
 5   transactiondescription  13497 non-null  object        
 6   initallocamount         13497 non-null  int64         
 7   reallocamount           13497 non-null  int64         
 8   trandate                13497 non-null  datetime64[ns]
 9   creditingstrategy       13497 non-null  object        
 10  adjustment              13497 non-null  float64       
 11  converge                13497 non-null  float64       
dtypes: datetime64[ns](2), float64(3), int64(2), ob

In [91]:
notional = notional.dropna(subset=['policynumber'], axis=0)

In [92]:
notional = notional.iloc[:,0:12]

In [93]:
to_upload(notional,"notional",get_seriatim_month)

start pushing policy data


100%|██████████| 1/1 [00:00<?, ?it/s]

time took to upload 5.6721765995025635
success


In [94]:
withdrawals = withdrawals.astype({"termdate" : "datetime64[ns]"})

In [95]:
to_upload(withdrawals,"withdrawals",get_seriatim_month)

start pushing policy data


100%|██████████| 1/1 [00:00<?, ?it/s]

time took to upload 5.204579591751099
success


In [96]:
premiums = premiums.dropna(subset=['policynumber'], axis=0)

In [97]:
premiums = premiums.iloc[:,0:17]

In [98]:
premiums['premiumrecognitionyyyymm']= premiums['premiumrecognitionyyyymm'].astype("int64")

In [99]:
premiums['terminateddate'] =premiums['terminateddate'].astype('datetime64[s]')
premiums['addtlpremrcvddate'] =premiums['addtlpremrcvddate'].astype('datetime64[s]')
premiums['deleteddate'] =premiums['deleteddate'].astype('datetime64[s]')
premiums['premiumrecognitionmonth'] =premiums['premiumrecognitionmonth'].astype('datetime64[s]')
premiums['plan'] = premiums['plan'].astype("string")

In [100]:
to_upload(premiums,"premiums",get_seriatim_month)

start pushing policy data


100%|██████████| 1/1 [00:00<?, ?it/s]

time took to upload 4.66284441947937
success


In [101]:
commissions

,policynumber,rider,reinsurancecode,issuedate,enterdate,description,agentnumber,premiumamount,commpercent,commissionamount,commissionrecognitionmonth,commissionrecognitionyyyymm,converge
0,TB00017522,None,AAJ,2021-04-14,2026-03-30,Commission Schedule changed because of age change,317660,-9194.15,0.020,-183.88,2026-03-30,202603,0.1
1,TB00017522,None,AAJ,2021-04-14,2026-03-30,Commission Schedule changed because of age change,1280460,-9194.15,0.095,-873.44,2026-03-30,202603,0.1
2,TB00017522,None,AAJ,2021-04-14,2026-03-30,Commission Earned,317660,9194.15,0.030,275.82,2026-03-30,202603,0.1
3,TB00017522,None,AAJ,2021-04-14,2026-03-30,Commission Earned,1280460,9194.15,0.090,827.47,2026-03-30,202603,0.1
4,TB00029653,None,AFR,2022-02-02,2026-03-10,Charge Back - Free Look,1400800,-70200.00,0.025,-1755.00,2026-03-10,202603,0.3
5,TB00029653,None,AFR,2022-02-02,2026-03-10,Charge Back - Free Look,1401770,-70200.00,0.085,-5967.00,2026-03-10,202603,0.3
6,TB00029653,None,AFR,2022-02-02,2026-03-10,Charge Back - Free Look,1225710,-70200.00,0.005,-351.00,2026-03-10,202603,0.3
7,TB00036634,None,AIQ,2023-03-15,2026-03-28,Additional Commission Earned,1265970,2995.16,0.061,182.70,2026-03-28,202603,0.1
8,TB00036634,None,AIQ,2023-03-15,2026-03-28,Additional Commission Earned,1221010,2995.16,0.004,11.98,2026-03-28,202603,0.1
9,TB00036634,None,AIQ,2023-03-15,2026-03-24,Additional Commission Earned,1265970,2203.63,0.061,134.42,2026-03-24,202603,0.1


In [102]:
commissions = commissions.iloc[:,0:13]

In [103]:
commissions = commissions.dropna(subset=['policynumber'],axis=0)

In [ ]:
#commissions.rename(columns={"policyid": "policynumber"}, inplace=True)

In [104]:
commissions['commissionrecognitionyyyymm'] = commissions['commissionrecognitionyyyymm'].astype("int64")

In [105]:
commissions['issuedate'] = commissions['issuedate'].astype("datetime64[ns]")
commissions['agentnumber'] = commissions['agentnumber'].astype("int64")
commissions['enterdate'] = commissions['enterdate'].astype("datetime64[ns]")
commissions['commissionrecognitionmonth'] = commissions['commissionrecognitionmonth'].astype("datetime64[ns]")

In [106]:
commissions = commissions.drop(['rider'], axis=1)

In [107]:
to_upload(commissions,"commissions",get_seriatim_month)

start pushing policy data


100%|██████████| 1/1 [00:00<?, ?it/s]

time took to upload 3.775377035140991
success


In [108]:
deaths

,policynumber,rider,termdate,reinsurancecode,totaldeath,converge
0,T000005790,None,2026-03-12,AC,1.076821e+05,0.10
1,T000006715,None,2026-03-25,AC,2.948679e+04,0.10
2,T000007200,None,2026-03-19,AC,5.024441e+04,0.10
3,T000007288,None,2026-03-03,AD,1.002593e+05,0.10
4,T000007610,None,2026-03-26,AAL,3.630612e+04,0.10
5,T000009627,None,2026-03-31,ACB,2.868214e+05,0.30
6,T000009878,None,2026-03-30,AFM,4.069087e+04,0.30
7,T000005789,None,2026-03-12,AC,3.292593e+05,0.10
8,T000011520,None,2026-03-31,AHI,2.315594e+04,0.10
9,T000006505,None,2026-03-27,AC,3.036949e+04,0.10


In [109]:
deaths = deaths.rename(columns ={"reinsurancecode" : "reins"})

In [110]:
deaths['converge'] = deaths['converge'].astype("float64")
deaths['reins'] = deaths['reins'].astype("string")

In [111]:
deaths = deaths.iloc[:,0:5]

In [112]:
deaths = deaths.dropna(subset=['policynumber'],axis=0)

In [113]:
deaths = deaths.drop(['rider'], axis=1)

In [114]:
to_upload(deaths,"deaths",get_seriatim_month)

start pushing policy data


100%|██████████| 1/1 [00:00<?, ?it/s]

time took to upload 3.7736759185791016
success


In [115]:
df_test = pd.DataFrame()

In [116]:
#Testing
def test(table, set_month, product):
    print('Startin the Test for', set_month)
    #product teton or denali 
    df = pd.DataFrame()
    if 'Teton' in product:
        product = 'T%'
    elif 'Denali' in product:
        product = 'D%'
    print('And Product  =', product)
    
    print('Table Name: ', table)
    print('Query running: ')
    if table == 'policy':    
        query = 'SELECT count(*) from `denali.policy` WHERE set_month="'+set_month+'" AND policynumber like "'+product+'"'
        excel_result=policy.policynumber.count()
    elif table =='seriatim':
        query = 'SELECT sum(reserves2) from `denali.seriatim_values` WHERE set_month="'+set_month+'" AND policynumber like "'+product+'"'
        excel_result=seriatim.reserves2.sum()
    elif table == 'premiums':
        query = 'SELECT sum(totaladdtlpremium) from `denali.premiums` WHERE CAST(premiumrecognitionyyyymm as STRING)="'+set_month+'" AND policynumber like "'+product+'"'
        excel_result=premiums.totaladdtlpremium.sum()
    elif table == 'withdrawals':
        query = 'SELECT sum(fullsurrenders) from `denali.withdrawals` WHERE set_month="'+set_month+'" AND policynumber like "'+product+'"'
        excel_result = withdrawals.fullsurrenders.sum()
    elif table == 'notional':
        query = 'SELECT sum(reallocamount) from `denali.notional` WHERE FORMAT_TIMESTAMP("%Y%m", trandate)="'+set_month+'" AND policynumber like "'+product+'"'
        excel_result = notional.reallocamount.sum()
    elif table == 'commissions':
        query = 'SELECT sum(commissionamount) from `denali.commissions` WHERE CAST(commissionrecognitionyyyymm as STRING)="'+set_month+'" AND policynumber like "'+product+'"'
        excel_result = commissions.commissionamount.sum()
    elif table == 'deaths':
        query = 'SELECT sum(totaldeath) from `denali.deaths` WHERE set_month="'+set_month+'" AND policynumber like "'+product+'"'
        excel_result = deaths.totaldeath.sum()
    print(query)
    job = client.query(query)  
    for i in job.result():
        result = i[0]
        
    #Comparing with the Excel file

    print(type(result))
    print(type(excel_result))
    print('starts check here')
   
    if round(float(result),2) == round(float(excel_result), 2):
        print("Database Result: ", round(float(result),2))
        print("Excel Result: ", round(float(excel_result),2))
        test_result = 'Pass'
    else:
        print("Database Result: ", result)
        print("Excel Result: ", excel_result)
        test_result = 'Fail'
    print(test_result)
    print('-----------------')
    return test_result

In [117]:
test_fields = ['policy', 'seriatim', 'premiums', 'withdrawals', 'notional', 'commissions', 'deaths']

for i in test_fields:
    test(i, get_seriatim_month, file_path)
    



Startin the Test for 202603
And Product  = T%
Table Name:  policy
Query running: 
SELECT count(*) from `denali.policy` WHERE set_month="202603" AND policynumber like "T%"
<class 'int'>
<class 'numpy.int64'>
starts check here
Database Result:  48805.0
Excel Result:  48805.0
Pass
-----------------
Startin the Test for 202603
And Product  = T%
Table Name:  seriatim
Query running: 
SELECT sum(reserves2) from `denali.seriatim_values` WHERE set_month="202603" AND policynumber like "T%"
<class 'float'>
<class 'numpy.float64'>
starts check here
Database Result:  1833565397.94
Excel Result:  1833565397.94
Pass
-----------------
Startin the Test for 202603
And Product  = T%
Table Name:  premiums
Query running: 
SELECT sum(totaladdtlpremium) from `denali.premiums` WHERE CAST(premiumrecognitionyyyymm as STRING)="202603" AND policynumber like "T%"
<class 'float'>
<class 'numpy.float64'>
starts check here
Database Result:  -1610420.07
Excel Result:  -1610420.07
Pass
-----------------
Startin the Tes

In [118]:
set_month = get_seriatim_month
import sys

In [119]:
#Importing AVRF analysis


sys.path.append('avrf_analysis_silac.py')
import avrf_analysis_silac
# Trigger the AVRF analysis

In [120]:
avrf_analysis_silac.run_avrf_analysis(set_month, product)

check set_month value: 202603
BigQuery client initialized successfully!
Running AVRF analysis for set_month: 
 202603 T%
difference is more then threshold amount:
AVRF result exported and Complete


In [121]:
query_report = f'''
UPDATE `denali.policy` p1
SET p1.reported_date = (SELECT MIN(p2.set_month) from `denali.policy` p2 WHERE p2.policynumber = p1.policynumber)
WHERE p1.reported_date IS NULL
'''
job = client.query(query_report) 

### Run this After the upload for Both Denali and Teton

In [123]:
sys.path.append('../actuarial-pipelines/reconciliations/silac/')
from reconciliation import run_reconciliation
run_reconciliation(set_month)

Starting the program 

Running reconciliation for SILAC 202603
SELECT CAST(sum(w.internalreissues*(SELECT max(sv.converge) FROM `denali.seriatim_values` AS sv   WHERE sv.policynumber = w.policynumber group by w.policynumber) ) as INT) as internalreissues_sum  FROM `denali.withdrawals` w WHERE set_month = "202603" AND w.policynumber LIKE "D%"
{'product': 'Denali', 'fieldname': 'internalreissues', 'total': 0}
-----------------------------------------------
SELECT CAST(sum(w.fullsurrenders*(SELECT max(sv.converge) FROM `denali.seriatim_values` AS sv   WHERE sv.policynumber = w.policynumber group by w.policynumber) ) as INT) as fullsurrenders_sum  FROM `denali.withdrawals` w WHERE set_month = "202603" AND w.policynumber LIKE "D%"
{'product': 'Denali', 'fieldname': 'fullsurrenders', 'total': 2643287}
-----------------------------------------------
SELECT CAST(sum(w.partialwithdrawals*(SELECT max(sv.converge) FROM `denali.seriatim_values` AS sv   WHERE sv.policynumber = w.policynumber group 

,product,fieldname,total
0,Denali,internalreissues,0
1,Denali,fullsurrenders,2643287
2,Denali,partialwithdrawals,1017021
3,Denali,cancellationamount,0
4,Denali,rmdwithdrawals,335359
5,Denali,otherwithdrawals,0
6,Denali,lifetimewithdrawals,246238
7,Denali,homecare,4277
8,Denali,nursinghome,66387
9,Denali,terminalillness,51179


In [124]:

# Add directory containing LDTI.py
sys.path.append('../actuarial-pipelines/ldti/')

# Import the function
from LDTI import main_query_run

# Trigger the AVRF analysis
main_query_run("SILAC")

Starting LDTI run for : SILAC

    WITH
    min_set_month AS (
      SELECT
      distinct
        policynumber,
        MIN(set_month) AS first_set_month,
        MIN(LEFT(reported_date,4)) as report_year,
      FROM `denali.policy`
      WHERE policynumber like 'D%'
      GROUP BY policynumber
    ),
    inforce_list as (
      select sv.set_month, LEFT(p.reported_date,4) as report_year, count(distinct sv.policynumber) as ct_inforce from `denali.seriatim_values` sv
      JOIN `denali.policy` p ON p.policynumber = sv.policynumber AND sv.set_month = p.set_month and sv.creditstrategy = p.creditstrategy
      WHERE sv.policynumber like 'D%' and totalpolicyav > 0
      GROUP by sv.set_month, LEFT(p.reported_date,4) 
      ORDER by sv.set_month, report_year
    )

    SELECT
      i.set_month,
      CAST(i.report_year as INT64) as report_year,
      i.ct_inforce,
      COUNT(m.policynumber) AS new_issued_policies
    FROM inforce_list i
    LEFT JOIN min_set_month m
      ON m.first_set_mo